In [10]:
import sys

from sklearn.model_selection import PredefinedSplit
sys.path.append("..")

import pandas as pd
import mlflow
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, f1_score
from src.temizle import Temizleyici

train = pd.read_csv("../data/processed/tr_train.csv")
test = pd.read_csv("../data/processed/tr_test.csv")
train = train[train["belirsiz"] != "e"]

X_train, y_train = train["mesaj"], train["ikili"]
X_test, y_test = test["mesaj"], test["ikili"]
cv = PredefinedSplit(train["fold"])
spam_f1 = make_scorer(f1_score, pos_label="spam")
print(X_train.shape, y_test.shape)



(651,) (116,)


In [11]:
import re
from collections import Counter
from keras.utils import pad_sequences
from src.temizle import temizle

DESEN = r"\[[A-ZÇĞİÖŞÜ]+\]|\w+|[!%?:]"

def tokenle(metin):
    return re.findall(DESEN, temizle(metin))

def sozluk_kur(metinler, min_df=2):
    sayac = Counter(k for m in metinler for k in set(tokenle(m)))
    kelimeler = [k for k, n in sayac.items() if n >= min_df]
    return {k: i + 2 for i, k in enumerate(kelimeler)}

def sayiya(metinler, sozluk, uzunluk=50):
    diziler = [[sozluk.get(k, 1) for k in tokenle(m)] for m in metinler]
    return pad_sequences(diziler, maxlen=uzunluk, padding="post", truncating="post")

sozluk = sozluk_kur(X_train)
print(len(sozluk))
print(tokenle(X_train.iloc[0]))
print(sayiya([X_train.iloc[0]], sozluk)[0][:15])

1072
['0', '0', 'arasi', 'halisaya', 'aldik', 'gelcekmsn']
[3 3 2 1 1 1 0 0 0 0 0 0 0 0 0]


In [20]:
def model_kur(sozluk_boyu, uzunluk=50):
    m = keras.Sequential([
        keras.layers.Input(shape=(uzunluk,)),
        keras.layers.Embedding(sozluk_boyu + 2, 32, mask_zero=True),
        keras.layers.Bidirectional(keras.layers.LSTM(32)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

model_kur(len(sozluk)).summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 32)         │        34,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,073 (199.50 KB)

 Trainable params: 51,073 (199.50 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
import numpy as np

AYAR = {"model": "BiLSTM", "token": "kelime", "min_df": 2, "uzunluk": 50,
        "embed": 32, "lstm": 32, "dropout": 0.3, "epoch": 15, "batch": 32}

def model_kur(sozluk_boyu, a):
    m = keras.Sequential([
        keras.layers.Input(shape=(a["uzunluk"],)),
        keras.layers.Embedding(sozluk_boyu + 2, a["embed"], mask_zero=True),
        keras.layers.Bidirectional(keras.layers.LSTM(a["lstm"])),
        keras.layers.Dropout(a["dropout"]),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy")
    return m

y_bin = (y_train == "spam").astype(int).to_numpy()
X_arr = X_train.to_numpy()
katlar = train["fold"].to_numpy()

def bilstm_cv(a):
    skorlar = []
    for k in range(5):
        egit, dogr = katlar != k, katlar == k
        sz = sozluk_kur(X_arr[egit], a["min_df"])
        Xe = sayiya(X_arr[egit], sz, a["uzunluk"])
        Xd = sayiya(X_arr[dogr], sz, a["uzunluk"])
        keras.utils.set_random_seed(42)
        m = model_kur(len(sz), a)
        m.fit(Xe, y_bin[egit], epochs=a["epoch"], batch_size=a["batch"], verbose=0)
        tahmin = (m.predict(Xd, verbose=0).ravel() > 0.5).astype(int)
        skorlar.append(f1_score(y_bin[dogr], tahmin))
        print(f"fold {k}: {skorlar[-1]:.3f}")
    return np.array(skorlar)

with mlflow.start_run(run_name="bilstm_kelime"):
    skor = bilstm_cv(AYAR)
    mlflow.log_params(AYAR)
    mlflow.log_metrics({"cv_f1": skor.mean(), "cv_f1_std": skor.std()})
    print(skor.mean().round(3), skor.round(3))

fold 0: 0.984
fold 1: 0.983
fold 2: 0.894
fold 3: 0.957
fold 4: 0.992
0.962 [0.984 0.983 0.894 0.957 0.992]


In [22]:
from sklearn.metrics import precision_score, recall_score

def olc(y_gercek, y_tahmin):
    return {"test_f1": f1_score(y_gercek, y_tahmin),
            "test_p": precision_score(y_gercek, y_tahmin),
            "test_r": recall_score(y_gercek, y_tahmin)}

y_test_bin = (y_test == "spam").astype(int).to_numpy()
sonuclar = {}

mlp.fit(X_train, y_train)
sonuclar["MLP harf"] = olc(y_test_bin, (mlp.predict(X_test) == "spam").astype(int))

sz = sozluk_kur(X_arr, AYAR["min_df"])
keras.utils.set_random_seed(42)
bl = model_kur(len(sz), AYAR)
bl.fit(sayiya(X_arr, sz, AYAR["uzunluk"]), y_bin, epochs=AYAR["epoch"], batch_size=AYAR["batch"], verbose=0)
olasilik = bl.predict(sayiya(X_test.to_numpy(), sz, AYAR["uzunluk"]), verbose=0).ravel()
sonuclar["BiLSTM kelime"] = olc(y_test_bin, (olasilik > 0.5).astype(int))

with mlflow.start_run(run_name="test_gun6"):
    for ad, s in sonuclar.items():
        mlflow.log_metrics({f"{ad.replace(' ', '_')}_{k}": v for k, v in s.items()})
        print(ad, {k: round(v, 3) for k, v in s.items()})

MLP harf {'test_f1': 0.981, 'test_p': 0.981, 'test_r': 0.981}
BiLSTM kelime {'test_f1': 0.971, 'test_p': 1.0, 'test_r': 0.944}


In [12]:
#MLFLOW

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline


mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("tr_ikili")

model = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False)),
    ("model", LinearSVC(C=1, random_state=55)),
])

with mlflow.start_run(run_name="svm_char_wb"):
    skor = cross_val_score(model, X_train, y_train, cv=cv, scoring=spam_f1)
    mlflow.log_params({"model": "LinearSVC", "feature": "char_wb 3-5", "C": 1})
    mlflow.log_metrics({"cv_f1": skor.mean(), "cv_f1_std": skor.std()})
    print(skor.mean().round(3))

0.971


In [13]:
def parametreler(model):
    p = model.get_params()
    return {
        "model": type(p["model"]).__name__,
        "analyzer": p["vek__analyzer"],
        "ngram_range": str(p["vek__ngram_range"]),
        "C": p["model__C"],
    }

print(parametreler(model))

{'model': 'LinearSVC', 'analyzer': 'char_wb', 'ngram_range': '(3, 5)', 'C': 1}


In [14]:
for md in [1, 2]:
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False, min_df=md)
    v.fit(Temizleyici().transform(X_train))
    print(f"min_df={md}: {len(v.vocabulary_)} parça")

min_df=1: 25076 parça
min_df=2: 12616 parça


In [15]:
def parametreler(model):
    p = model.get_params()
    sonuc = {
        "model": type(p["model"]).__name__,
        "analyzer": p["vek__analyzer"],
        "ngram_range": str(p["vek__ngram_range"]),
        "min_df": p["vek__min_df"],
    }
    for ad, deger in p.items():
        if ad.startswith("model__") and isinstance(deger, (int, float, str, tuple)):
            sonuc[ad] = str(deger)
    return sonuc

In [16]:
from sklearn.neural_network import MLPClassifier

mlp = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False, min_df=2)),
    ("model", MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=300, random_state=42)),
])

with mlflow.start_run(run_name="mlp_char_wb"):
    skor = cross_val_score(mlp, X_train, y_train, cv=cv, scoring=spam_f1)
    mlflow.log_params(parametreler(mlp))
    mlflow.log_metrics({"cv_f1": skor.mean(), "cv_f1_std": skor.std()})
    print(skor.mean().round(3), skor.round(3))

0.974 [0.953 0.968 0.992 0.975 0.983]


In [17]:
from sklearn.model_selection import cross_validate

for ad, m in [("SVM", model), ("MLP", mlp)]:
    s = cross_validate(m, X_train, y_train, cv=cv, scoring=spam_f1, return_train_score=True)
    print(f"{ad}: train {s['train_score'].mean():.3f}  cv {s['test_score'].mean():.3f}  fark {s['train_score'].mean() - s['test_score'].mean():.3f}")

SVM: train 1.000  cv 0.971  fark 0.029
MLP: train 1.000  cv 0.974  fark 0.026


In [18]:
#BILSTM
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
print(keras.__version__, keras.backend.backend())

3.15.1 torch
